In [1]:
from src import jobs

jobs.all.run()

2026-04-03 14:34:52.975 | INFO | stagefy | Running SparkConnection.__build_spark_session method...
2026-04-03 14:34:52.976 | DEBUG | stagefy | module='stagefy.connections.spark', method='SparkConnection.__build_spark_session', args=(SparkConnection(config={'spark.sql.session.timeZone': 'America/Sao_Paulo'}),), kwargs={}
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 14:34:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-04-03 14:34:55.68 | INFO | stagefy | Running SparkConnection.__set_spark_session_configuration method...
2026-04-03 14:34:55.69 | DEBUG | stagefy | module='stagefy.connections.spark', method='SparkConnection.__set_spark_session_configuration', args=(SparkConnection(config={'spark.sql.session.timeZone': 'America/Sao_Paulo'}),), kwargs={}
2026-04-03 14:34:55.470 | INFO | stagefy | Running Process.ru

In [2]:
from stagefy.connections import SparkConnection
import pyspark.sql.functions as f

connection = SparkConnection()

2026-04-03 14:35:01.32 | INFO | stagefy | Running SparkConnection.__build_spark_session method...
2026-04-03 14:35:01.33 | DEBUG | stagefy | module='stagefy.connections.spark', method='SparkConnection.__build_spark_session', args=(SparkConnection(config=None),), kwargs={}
2026-04-03 14:35:01.37 | INFO | stagefy | Running SparkConnection.__set_spark_session_configuration method...
2026-04-03 14:35:01.38 | DEBUG | stagefy | module='stagefy.connections.spark', method='SparkConnection.__set_spark_session_configuration', args=(SparkConnection(config=None),), kwargs={}


In [3]:
connection.spark.catalog.listTables()

[Table(name='pyspark_data_reconciliation__calc__agg_reconciliation_summary', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='pyspark_data_reconciliation__calc__erp_orders', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='pyspark_data_reconciliation__calc__gateway_transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='pyspark_data_reconciliation__calc__recon_order_result', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='pyspark_data_reconciliation__stg__erp_orders', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='pyspark_data_reconciliation__stg__erp_orders_raw', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='pyspark_data_reconciliation__stg__gateway_transactions', 

In [4]:
for view in connection.spark.catalog.listTables():
    dataframe = connection.spark.table(view.name)
    print('View: ' + view.name)
    print('Total records: ' + str(dataframe.count()))
    dataframe.printSchema()
    dataframe.show(truncate=False)

View: pyspark_data_reconciliation__calc__agg_reconciliation_summary
Total records: 1
root
 |-- total_orders: long (nullable = false)
 |-- matched_count: long (nullable = true)
 |-- divergent_count: long (nullable = true)
 |-- missing_in_erp_count: long (nullable = true)
 |-- missing_in_gateway_count: long (nullable = true)
 |-- amount_mismatch_count: long (nullable = true)
 |-- status_mismatch_count: long (nullable = true)
 |-- matched_rate: double (nullable = true)

+------------+-------------+---------------+--------------------+------------------------+---------------------+---------------------+------------+
|total_orders|matched_count|divergent_count|missing_in_erp_count|missing_in_gateway_count|amount_mismatch_count|status_mismatch_count|matched_rate|
+------------+-------------+---------------+--------------------+------------------------+---------------------+---------------------+------------+
|8           |2            |6              |2                   |2                  

In [5]:
status_condition = [
    ('matched_count', f.col('reconciliation_status') == 'MATCHED'),
    ('divergent_count', f.col('reconciliation_status') != 'MATCHED'),
    ('missing_in_erp_count', f.col('reconciliation_status') == 'MISSING_IN_ERP'),
    (
        'missing_in_gateway_count',
        f.col('reconciliation_status') == 'MISSING_IN_GATEWAY',
    ),
    ('amount_mismatch_count', f.col('reconciliation_status') == 'AMOUNT_MISMATCH'),
    ('status_mismatch_count', f.col('reconciliation_status') == 'STATUS_MISMATCH')
]

dataframe = connection.spark.table(
    'pyspark_data_reconciliation__calc__recon_order_result'
).select(
    f.count(f.lit(1)).alias('total_orders'),
    *[
        f.sum(f.when(condition, 1).otherwise(0)).alias(alias)
        for alias, condition in status_condition
    ],
).select(
    f.col('total_orders'),
    *[f.col(column) for column, _ in status_condition],
    ((f.col('matched_count') / f.col('total_orders')) * 100).alias('matched_rate'),
)

In [6]:
dataframe.printSchema()

root
 |-- total_orders: long (nullable = false)
 |-- matched_count: long (nullable = true)
 |-- divergent_count: long (nullable = true)
 |-- missing_in_erp_count: long (nullable = true)
 |-- missing_in_gateway_count: long (nullable = true)
 |-- amount_mismatch_count: long (nullable = true)
 |-- status_mismatch_count: long (nullable = true)
 |-- matched_rate: double (nullable = true)



In [7]:
dataframe.show(truncate=False)

+------------+-------------+---------------+--------------------+------------------------+---------------------+---------------------+------------+
|total_orders|matched_count|divergent_count|missing_in_erp_count|missing_in_gateway_count|amount_mismatch_count|status_mismatch_count|matched_rate|
+------------+-------------+---------------+--------------------+------------------------+---------------------+---------------------+------------+
|8           |2            |6              |2                   |2                       |1                    |1                    |25.0        |
+------------+-------------+---------------+--------------------+------------------------+---------------------+---------------------+------------+

